In [1]:
import example_loader as el
import gurobipy as gp
import gurobi_utils as gu
import miplib_loader as ml
from tqdm.auto import tqdm

In [20]:
# ways forward: maybe the +1 is wrong; I'm reducing vars (as I don't have the slacks)
# the negated rows may always be slacks on constraint >=
# is there one row or column that I must always negate in order for this to work?
# should I try the <= standardization once more?

import importlib
importlib.reload(gu)

def run_validation(instances):
    for instance in instances:
        model: gp.Model = instance.as_gurobi_model()
        print("Running model:", model.ModelName)
        model.params.LogToConsole = 0
        model.params.Method = 1
        gu.standardize_lt_to_gt(model)
        int_vars, int_idx = gu.relax_int_or_bin_to_continuous(model)
        model.optimize()
        assert model.Status == gp.GRB.OPTIMAL
        basis = gu.read_basis(model)
        tableau, col_to_var, negated_rows = gu.read_tableau(model, basis)
        with tqdm(range(tableau.shape[1])) as bar:  # weird bug where it doesn't draw the last item...
            failures = gu.validate_corner(model, basis, tableau, col_to_var, iter(bar))
        print("   Failures:", failures)

# test the cuts on simple examples:
run_validation(list(el.get_instances().values()))

Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Running model: 2D from bottom
   Negated 2 constraints on 2D from bottom
   Relaxed 2 variables on 2D from bottom


  0%|          | 0/2 [00:00<?, ?it/s]

   Failures: 0
Running model: 2D from top
   Negated 0 constraints on 2D from top
   Relaxed 2 variables on 2D from top


  0%|          | 0/2 [00:00<?, ?it/s]

   Failures: 0
Running model: 2D from bottom (manual slacks)
   Negated 0 constraints on 2D from bottom (manual slacks)
   Relaxed 2 variables on 2D from bottom (manual slacks)


  0%|          | 0/4 [00:00<?, ?it/s]

   Failures: 0
Running model: Book example 6.3
   Negated 3 constraints on Book example 6.3
   Relaxed 3 variables on Book example 6.3


  0%|          | 0/3 [00:00<?, ?it/s]

   Failures: 0


In [10]:
miplib_instances = ml.get_instances()
miplib_subset = [miplib_instances['air05'], miplib_instances['markshare2'], miplib_instances['mas76']]  # drayage-25-23
run_validation(miplib_subset)

Read MPS format model from file mip2017_benchmark/revised-submissions/miplib2010_publically_available/instances/air05.mps.gz
Reading time = 0.02 seconds
air05: 426 rows, 7195 columns, 52121 nonzeros
Running model: air05
   Negated 0 constraints on air05
   Relaxed 7195 variables on air05


  0%|          | 0/7195 [00:00<?, ?it/s]

   Failures: 0
Read MPS format model from file mip2017_benchmark/revised-submissions/miplib2010_publically_available/instances/markshare2.mps.gz
Reading time = 0.01 seconds
markshare2: 7 rows, 74 columns, 434 nonzeros
Running model: markshare2
   Negated 0 constraints on markshare2
   Relaxed 60 variables on markshare2


  0%|          | 0/74 [00:00<?, ?it/s]

   Failures: 0
Read MPS format model from file mip2017_benchmark/revised-submissions/miplib2010_publically_available/instances/mas76.mps.gz
Reading time = 0.01 seconds
mas76: 12 rows, 151 columns, 1640 nonzeros
Running model: mas76
   Negated 1 constraints on mas76
   Relaxed 150 variables on mas76


  0%|          | 0/151 [00:00<?, ?it/s]

   Failures: 0
